# Tutorial: Phase 1 Model Training

Audience:
- Teams training the classical ML baseline before moving to deep learning.

Prerequisites:
- Familiarity with train/test split, feature matrices, and standard evaluation metrics.

Objective:
- Run the full Phase 1 pipeline, inspect the cached feature table, and compare SVM and Random Forest results.


## Outline

1. Setup and configuration
2. Run the full Phase 1 pipeline if needed
3. Inspect cached feature matrix and labels
4. Compare model metrics
5. Review class balance in the cached features


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 140)
SEED = 42


/Users/vaishnavverma/.matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/matplotlib-x21mos5f because there was an issue with the default path (/Users/vaishnavverma/.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Matplotlib is building the font cache; this may take a moment.


In [2]:
CANDIDATE_DATA_DIRS = [
    PROJECT_ROOT / 'dataset',
    PROJECT_ROOT / 'data' / 'raw' / 'Multi-Class Skin Condition Image Dataset (MSC-6)',
]
DATA_DIR = next((path for path in CANDIDATE_DATA_DIRS if path.exists()), CANDIDATE_DATA_DIRS[0])
CANDIDATE_OUTPUT_DIRS = [
    PROJECT_ROOT / 'outputs' / 'phase1_baseline_msc6',
    PROJECT_ROOT / 'outputs' / 'phase1_baseline',
]
OUTPUT_DIR = next((path for path in CANDIDATE_OUTPUT_DIRS if path.exists()), CANDIDATE_OUTPUT_DIRS[0])
IMAGE_SIZE = (224, 224)
TEST_SIZE = 0.2
RANDOM_STATE = 42
SAMPLE_FEATURE_ROWS = 64
SIZE_SCAN_LIMIT = 1000
print('Using DATA_DIR:', DATA_DIR)
print('Using OUTPUT_DIR:', OUTPUT_DIR)

from src.skin_analysis.pipeline import run_phase1_pipeline


Using DATA_DIR: /Users/vaishnavverma/Downloads/Hybrid-Dermatologist/data/raw/Multi-Class Skin Condition Image Dataset (MSC-6)
Using OUTPUT_DIR: /Users/vaishnavverma/Downloads/Hybrid-Dermatologist/outputs/phase1_baseline_msc6


/Users/vaishnavverma/Downloads/Hybrid-Dermatologist/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
metrics_path = OUTPUT_DIR / 'metrics_summary.csv'
feature_cache_path = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset.csv'

if not metrics_path.exists() or not feature_cache_path.exists():
    results = run_phase1_pipeline(
        data_dir=DATA_DIR,
        output_dir=OUTPUT_DIR,
        image_size=IMAGE_SIZE,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    metrics_df = results['metrics_summary']
else:
    metrics_df = pd.read_csv(metrics_path)

features_df = pd.read_csv(feature_cache_path)
feature_columns = [column for column in features_df.columns if column.startswith('feature_')]
X = features_df[feature_columns].to_numpy()
y = features_df['label'].to_numpy()

print('X shape:', X.shape)
print('y shape:', y.shape)
display(metrics_df)


X shape: (8823, 154)
y shape: (8823,)


,model,accuracy,precision,recall,f1_score,average_method
0,random_forest,0.809065,0.810676,0.809065,0.806163,weighted
1,svm,0.785269,0.800390,0.785269,0.788009,weighted


In [4]:
class_counts = (
    features_df['display_label']
    .value_counts()
    .rename_axis('class_name')
    .reset_index(name='feature_rows')
)
display(class_counts)

plt.figure(figsize=(8, 5))
sns.barplot(data=class_counts, x='class_name', y='feature_rows', palette='Set2')
plt.title('Cached Feature Rows Per Class')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


,class_name,feature_rows
0,eczema,3133
1,normal,1240
2,acne,1185
3,rosacea,1108
4,wrinkles,1100
5,dark spots,1057


/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_58292/1305279312.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=class_counts, x='class_name', y='feature_rows', palette='Set2')
/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_58292/1305279312.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
plot_df = metrics_df.melt(id_vars=['model', 'average_method'], value_vars=['accuracy', 'precision', 'recall', 'f1_score'])
plt.figure(figsize=(8, 5))
sns.barplot(data=plot_df, x='variable', y='value', hue='model', palette='Set2')
plt.ylim(0.0, 1.0)
plt.xlabel('Metric')
plt.ylabel('Score')
plt.title('Phase 1 Model Comparison')
plt.tight_layout()
plt.show()

print('Artifacts saved to:', OUTPUT_DIR)


Artifacts saved to: /Users/vaishnavverma/Downloads/Hybrid-Dermatologist/outputs/phase1_baseline_msc6


/var/folders/gp/0qsbr0x11d315dt15fl17c940000gn/T/ipykernel_58292/960733153.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Notes

- The heavy lifting is done by the shared pipeline so the notebook stays usable on larger datasets.
- The cached `features_dataset.csv` gives you a reproducible `X` and `y` table for inspection without re-extracting features every time.
- Use the evaluation notebook to inspect confusion matrices and class-wise reports in detail.
